In [1]:
%pip install -q sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [15]:
documents = [
        "Employees are compensated on a bi-weekly basis through direct deposit.",       
        "Employees must submit a leave request for approval.", 
        "Company internet must be used for work-related tasks only.",
        "Company internet is a broadband internet.",
        "Employees can take an hour break.",
        "Interact with each employee with Respect"
]

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
doc_vectors = model.encode(documents)
query = "What’s the internet usage policy?"
query_vec = model.encode([query])[0]

similarities = model.similarity(query_vec, doc_vectors)
print(similarities)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tensor([[0.0820, 0.0975, 0.4476, 0.4798, 0.1132, 0.0443]])


In [16]:
import numpy as np
similarities = np.asarray(similarities).squeeze()
similarities

array([0.08198839, 0.09747534, 0.44755295, 0.47982496, 0.11316206,
       0.04429522], dtype=float32)

In [17]:
top_3_indices = np.argsort(similarities)[::-1][:3]
print(top_3_indices)
top_scores = similarities[top_3_indices]
top_scores

[3 2 4]


array([0.47982496, 0.44755295, 0.11316206], dtype=float32)

In [18]:
top_3_indices = np.argsort(similarities)[::-1][:3]
print(top_3_indices)
top_scores = similarities[top_3_indices]
top_docs = [documents[i] for i in top_3_indices]


print (top_docs)
context = ", ".join(top_docs)
context
print(f"User query: {query}")
print(f"Context: {context}")
response = ask_question_open_ai(query)
print(f"\n\nOpen AI Response: {response}")

[3 2 4]
['Company internet is a broadband internet.', 'Company internet must be used for work-related tasks only.', 'Employees can take an hour break.']
User query: What’s the internet usage policy?
Context: Company internet is a broadband internet., Company internet must be used for work-related tasks only., Employees can take an hour break.


Open AI Response: Internet usage policy based on the provided context:

- Internet type: Broadband.
- Allowed usage: Must be used for work-related tasks only.
- Breaks: Employees can take an hour break.


In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

my_client = OpenAI(api_key=my_api_key)
# my_client

def ask_question_open_ai(prompt):

    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create

    llm_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        # messages=[
        #     {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
        #     {"role": "user", "content": prompt}
        # ]
        messages=[
            {"role": "system", "content": '''
             You are an assistant who answers only based on the given context.
             '''},
            {"role": "user", "content": f"Context: {context}\n\n User Question: {query}"} 
        ]

    )
    return llm_response.choices[0].message.content  